In [ ]:
from typing import Dict, List, Any
from datetime import datetime, timedelta
import requests
import math

try:
    import cartopy.crs as ccrs
    import matplotlib.pyplot as plt
except Exception:  # pragma: no cover - cartopy optional
    ccrs = None
    plt = None


In [ ]:
def fetch_forecast(lat: float, lon: float, models: List[str]) -> Dict[str, List[Dict[str, Any]]]:
    """Fetch 48 hour wind forecasts for several weather models.

Args:
    lat: Latitude in decimal degrees.
    lon: Longitude in decimal degrees.
    models: List of model identifiers supported by Open-Meteo.

Returns:
    Mapping of each model name to a list of hourly records with time,
    wind speed, gust and direction. Values may be None if data is
    missing from the API.

Example:
    >>> fetch_forecast(54.1, 8.25, ["gfs", "icon"])
    {'gfs': [...], 'icon': [...]}"""
    result: Dict[str, List[Dict[str, Any]]] = {}
    base_url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "windspeed_10m,windgusts_10m,winddirection_10m",
        "forecast_days": 2,
        "windspeed_unit": "kn",
    }
    for model in models:
        params["models"] = model
        response = requests.get(base_url, params=params, timeout=10)
        response.raise_for_status()
        hourly = response.json()["hourly"]
        records: List[Dict[str, Any]] = []
        for i, ts in enumerate(hourly["time"][0:48]):
            records.append({
                "time": ts,
                "wind": hourly["windspeed_10m"][i],
                "gust": hourly["windgusts_10m"][i],
                "direction": hourly["winddirection_10m"][i],
            })
        result[model] = records
    return result



In [ ]:
def find_first_valid_record(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Return first record containing both wind and direction values.

Args:
    records: Sequence of forecast entries.

Returns:
    The first item with non-None wind and direction, or the original
    first record if none match.

Example:
    >>> find_first_valid_record([
    ...     {"wind": None, "direction": None},
    ...     {"wind": 5, "direction": 90}
    ... ])
    {'wind': 5, 'direction': 90}"""
    for record in records:
        if record.get("wind") is not None and record.get("direction") is not None:
            return record
    return records[0]



In [ ]:
def next_noon(now: datetime | None = None) -> datetime:
    """Return the upcoming UTC noon time."""
    now = now or datetime.utcnow()
    today_noon = now.replace(hour=12, minute=0, second=0, microsecond=0)
    return today_noon if now <= today_noon else today_noon + timedelta(days=1)


In [ ]:
def fetch_noon_wind(lat: float, lon: float, model: str) -> tuple[float | None, float | None]:
    """Fetch wind data for the upcoming UTC noon at a single location."""
    base_url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude': lat,
        'longitude': lon,
        'hourly': 'windspeed_10m,winddirection_10m',
        'models': model,
        'forecast_days': 2,
        'windspeed_unit': 'kn',
        'timezone': 'UTC',
    }
    response = requests.get(base_url, params=params, timeout=10)
    response.raise_for_status()
    hourly = response.json()['hourly']
    times = [datetime.fromisoformat(t) for t in hourly['time']]
    idx = min(range(len(times)), key=lambda i: abs(times[i] - next_noon()))
    return hourly['windspeed_10m'][idx], hourly['winddirection_10m'][idx]


In [ ]:
def fetch_wind_grid(lat: float, lon: float, model: str, size: int = 11):
    """Gather noon wind data on an evenly spaced grid around a point."""
    step = 2.0 / (size - 1)
    lats = [lat - 1 + step * i for i in range(size)]
    lons = [lon - 1 + step * j for j in range(size)]
    speeds = []
    directions = []
    for la in lats:
        row_s = []
        row_d = []
        for lo in lons:
            speed, direction = fetch_noon_wind(la, lo, model)
            row_s.append(speed if speed is not None else float("nan"))
            row_d.append(direction if direction is not None else float("nan"))
        speeds.append(row_s)
        directions.append(row_d)
    return lats, lons, speeds, directions


In [ ]:
def create_wind_map(lat: float, lon: float, model: str, output_path: str, size: int = 11) -> str:
    """Render a wind barb map using Cartopy for a grid around the point."""
    if ccrs is None or plt is None:
        raise RuntimeError("Cartopy and matplotlib are required for map rendering")
    lats, lons, speeds, directions = fetch_wind_grid(lat, lon, model, size)
    lon_grid = [lons for _ in lats]
    lat_grid = [[la for _ in lons] for la in lats]
    u = [[-s * math.sin(math.radians(d)) for s, d in zip(row_s, row_d)]
         for row_s, row_d in zip(speeds, directions)]
    v = [[-s * math.cos(math.radians(d)) for s, d in zip(row_s, row_d)]
         for row_s, row_d in zip(speeds, directions)]
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": ccrs.PlateCarree()})
    ax.set_extent([lon - 1, lon + 1, lat - 1, lat + 1])
    ax.coastlines()
    ax.barbs(lon_grid, lat_grid, u, v, length=5, pivot="middle")
    fig.savefig(output_path, bbox_inches="tight")
    plt.close(fig)
    return output_path


In [ ]:
def render_forecast_table(forecasts: Dict[str, List[Dict[str, Any]]], output_path: str) -> str:
    """Write forecast data to an SVG file containing plain text rows.

The table lists time, wind and gust values for each model. Missing numbers are shown as a dash.

Example:
    >>> render_forecast_table({'gfs': [{'time': '2024-01-01T00:00', 'wind': 10, 'gust': 12}]}, 'table.svg')
    'table.svg"""
    model_names = list(forecasts.keys())
    header = ["time"]
    for name in model_names:
        header.extend([f"{name} wind", f"{name} gust"])
    lines = [" ".join(header)]
    hours = len(next(iter(forecasts.values())))
    for i in range(min(48, hours)):
        row = [forecasts[model_names[0]][i]["time"]]
        for name in model_names:
            wind = forecasts[name][i]["wind"]
            gust = forecasts[name][i]["gust"]
            row.append(str(wind) if wind is not None else "-")
            row.append(str(gust) if gust is not None else "-")
        lines.append(" ".join(row))
    svg_lines = []
    for idx, text in enumerate(lines, start=1):
        svg_lines.append(f'<text x="0" y="{15 * idx}" font-size="12">{text}</text>')
    height = 15 * (len(lines) + 1)
    svg = (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="900" height="{height}">'
        + "".join(svg_lines)
        + "</svg>"
    )
    with open(output_path, "w", encoding="utf-8") as file:
        file.write(svg)
    return output_path

